# Sample non-dimensional planetary-geostrophic problem

This notebook specifies a PG problem with a broad zonal depth slope and an off-center Gaussian seamount, solves for surface pressure, samples the full 3D solution, writes VTK files, and plots the solution and diagnostic balances. The bathymetry remains strictly positive while varying in both horizontal directions, so the example exercises the full $H(x,y)$ API.

In [1]:
using Pkg
Pkg.activate(joinpath(@__DIR__, "..", "PlanetaryGeostrophy"))
using PlanetaryGeostrophy

  Activating project at `~/Library/CloudStorage/OneDrive-JohnsHopkins/Documents/Articles/Manuscripts/In prep/Haine_etal/WorkingResearch/PlanetaryGeostrophyAnalyticalModel/PlanetaryGeostrophy`


In [2]:
Lx, Ly = 2.0, 1.5
H0 = 1.0
boundary = rectangle_boundary(Lx, Ly)
parameters = PGParameters(0.02, 0.20; viscosity=1.0, diffusivity=1.0)

# A broad zonal slope plus a smooth off-center seamount exercises H(x, y).
depth_slope = 0.15
seamount_amplitude = 0.25
seamount_center = (-0.25Lx, 0.10Ly)
seamount_width = (0.25Lx, 0.30Ly)
seamount(x, y) = exp(
    -((x - seamount_center[1]) / seamount_width[1])^2 -
    ((y - seamount_center[2]) / seamount_width[2])^2
)
depth(x, y) = H0 * (
    1 + depth_slope * x / Lx - seamount_amplitude * seamount(x, y)
)
depth_x(x, y) = H0 * (
    depth_slope / Lx +
    2seamount_amplitude * (x - seamount_center[1]) /
    seamount_width[1]^2 * seamount(x, y)
)
depth_y(x, y) = H0 * (
    2seamount_amplitude * (y - seamount_center[2]) /
    seamount_width[2]^2 * seamount(x, y)
)

problem = PGProblem(
    parameters=parameters,
    depth=depth,
    depth_x=depth_x,
    depth_y=depth_y,
    coriolis=(x, y) -> 1.0 + 0.15y,
    coriolis_x=(x, y) -> 0.0,
    coriolis_y=(x, y) -> 0.15,
    wind_stress=(x, y) -> 1e-3 * cos(pi * y / Ly) + 0im,
    wind_stress_curl=(x, y) -> 1e-3 * pi / Ly * sin(pi * y / Ly),
    restoring_buoyancy=(x, y, z) -> 1e-2z + 2e-3 * sin(pi * x / Lx),
    surface_buoyancy=(x, y) -> 2e-3 * cos(pi * x / Lx) * cos(pi * y / Ly),
    boundary=boundary,
)

PGProblem{PGParameters{Float64}, typeof(depth), var"#29#30", var"#31#32", var"#33#34", var"#35#36", var"#37#38", var"#39#40", var"#41#42", typeof(depth_x), typeof(depth_y), Vector{Tuple{Float64, Float64}}, PlanetaryGeostrophy.var"#87#88"{Vector{Tuple{Float64, Float64}}}}(PGParameters{Float64}(0.02, 0.2, 1.0, 1.0), Main.depth, var"#29#30"(), var"#31#32"(), var"#33#34"(), var"#35#36"(), var"#37#38"(), var"#39#40"(), var"#41#42"(), Main.depth_x, Main.depth_y, [(-1.0, -0.75), (1.0, -0.75), (1.0, 0.75), (-1.0, 0.75)], PlanetaryGeostrophy.var"#87#88"{Vector{Tuple{Float64, Float64}}}([(-1.0, -0.75), (1.0, -0.75), (1.0, 0.75), (-1.0, 0.75)]), 1.0e-5)

In [ ]:
solution = solve_pg(
    problem;
    mesh_file=joinpath(@__DIR__, "sample_pg_pressure.msh"),
    mesh_resolution=0.08,
    overwrite_mesh=true,
)

In [ ]:
x = range(-0.99Lx / 2, 0.99Lx / 2; length=64)
y = range(-0.99Ly / 2, 0.99Ly / 2; length=64)
z = range(-H0, 0.0; length=64)
fields = sample_solution(solution, x, y, z)

In [ ]:
files = write_solution(fields, joinpath(@__DIR__, "sample_pg"))
figure = plot_solution(
    fields;
    sigma_level=0.5,
    output=joinpath(@__DIR__, "sample_pg_solution.png"),
)
display(files)
display(figure)

In [ ]:
# The direct analytical diagnostic quadratures are more expensive than field sampling.
diagnostic_stride = 2
diagnostic_x = x[1:diagnostic_stride:end]
diagnostic_y = y[1:diagnostic_stride:end]
diagnostics = compute_diagnostics(solution, diagnostic_x, diagnostic_y)
display(performance_summary(diagnostics))

In [ ]:
bottom_stress_figure = plot_bottom_stress(
    diagnostics;
    arrow_scale=0.70,
    output=joinpath(@__DIR__, "sample_pg_bottom_stress.png"),
)
transport_figure = plot_depth_integrated_current(
    diagnostics;
    arrow_scale=0.70,
    output=joinpath(@__DIR__, "sample_pg_transport.png"),
)
streamfunction_figure = plot_streamfunction(
    diagnostics;
    output=joinpath(@__DIR__, "sample_pg_streamfunction.png"),
)
diagnostic_figure = plot_diagnostics(
    diagnostics;
    arrow_scale=0.70,
    output=joinpath(@__DIR__, "sample_pg_diagnostics.png"),
)
display(bottom_stress_figure)
display(transport_figure)
display(streamfunction_figure)
display(diagnostic_figure)

## Depth-integrated momentum balance

Compute the terms corresponding to equations (32)--(33), together with bottom pressure $p_b$ and baroclinic potential energy $\chi$. The API consistently uses conventional oceanographic buoyancy, $\partial_z p=b$, with $p=p_s-\int_z^0 b\,dz$.

In [ ]:
dim = compute_dim_diagnostics(solution, diagnostics)
display(performance_summary(dim))

In [ ]:
dim_balance_figure = plot_dim_balance(
    dim;
    output=joinpath(@__DIR__, "sample_pg_dim_balance.png"),
)
chi_figure = plot_baroclinic_potential_energy(
    dim;
    output=joinpath(@__DIR__, "sample_pg_chi.png"),
)
bottom_pressure_figure = plot_bottom_pressure(
    dim;
    output=joinpath(@__DIR__, "sample_pg_bottom_pressure.png"),
)
display(dim_balance_figure)
display(chi_figure)
display(bottom_pressure_figure)

## Barotropic vorticity balance

Compute and plot the transport-divergence, planetary-vorticity advection, bottom-pressure torque, boundary-stress curls, and closure residual in equation (36).

In [ ]:
barotropic_vorticity = compute_barotropic_vorticity_diagnostics(
    solution, diagnostics,
)
display(performance_summary(barotropic_vorticity))

barotropic_vorticity_figure = plot_barotropic_vorticity_balance(
    barotropic_vorticity;
    output=joinpath(@__DIR__, "sample_pg_barotropic_vorticity_balance.png"),
)
display(barotropic_vorticity_figure)

## Depth-integrated vorticity balance

Compute and plot the planetary-vorticity advection, bottom vortex stretching, bottom-stress curl, surface-stress curl, and closure residual in equation (41).

In [ ]:
vorticity = compute_vorticity_diagnostics(solution, diagnostics)
display(performance_summary(vorticity))

vorticity_figure = plot_vorticity_balance(
    vorticity;
    output=joinpath(@__DIR__, "sample_pg_vorticity_balance.png"),
)
display(vorticity_figure)

## Geostrophic-contour vorticity balance

Compute and plot geostrophic-contour advection, JEBAR, the depth-weighted bottom- and surface-stress curls, and the closure residual in equation (43).

In [ ]:
gcvb = compute_geostrophic_contour_vorticity_diagnostics(
    solution, diagnostics,
)
display(performance_summary(gcvb))

gcvb_figure = plot_geostrophic_contour_vorticity_balance(
    gcvb;
    output=joinpath(@__DIR__, "sample_pg_geostrophic_contour_vorticity_balance.png"),
)
display(gcvb_figure)

## Transport-divergence vorticity balance

Compute and plot the bottom-pressure torque, baroclinic torque, Coriolis-weighted bottom- and surface-stress curls, and closure residual in equation (46).

In [ ]:
tdvb = compute_transport_divergence_vorticity_diagnostics(
    solution, diagnostics,
)
display(performance_summary(tdvb))

tdvb_figure = plot_transport_divergence_vorticity_balance(
    tdvb;
    output=joinpath(@__DIR__, "sample_pg_transport_divergence_vorticity_balance.png"),
)
display(tdvb_figure)

## Surface-pressure equation coefficients

Compute and plot the advection velocity, diffusivity, and separate baroclinic and surface-stress source fields in the surface-pressure equation (74).

In [ ]:
pressure_equation = compute_surface_pressure_equation_diagnostics(
    solution, diagnostics,
)
display(performance_summary(pressure_equation))

pressure_equation_figure = plot_surface_pressure_equation_diagnostics(
    pressure_equation;
    arrow_scale=0.70,
    output=joinpath(@__DIR__, "sample_pg_surface_pressure_equation.png"),
)
display(pressure_equation_figure)